# ADNI MRI SPM — HIZLANDIRILMIŞ VERSİYON

**Mevcut durum:** 400 dosya bitti (Drive'da). Kalan: 3578 dosya.

## 3 TAB PLANI — Aynı anda aç, hepsini çalıştır

| Tab | BATCH_START | BATCH_END | Dosya |
|-----|------------|-----------|-------|
| 1   | 400        | 1000      | 600   |
| 2   | 1000       | 1600      | 600   |
| 3   | 1600       | 2200      | 600   |

**Sonra (3 session daha):**

| Tab | BATCH_START | BATCH_END |
|-----|------------|-----------|
| 1   | 2200       | 2800      |
| 2   | 2800       | 3400      |
| 3   | 3400       | 3978      |

---
**Optimizasyonlar:** N_JOBS 8→12 · samp 3→4 · bias write kapalı

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
%%bash
set -e
SPM_ZIP="/content/drive/MyDrive/SPM/spm_standalone_25.01.02_Linux.zip"
rm -rf /content/spm && mkdir -p /content/spm
cp "$SPM_ZIP" /content/spm/
cd /content/spm && unzip -q spm_standalone_25.01.02_Linux.zip
chmod +x /content/spm/spm_standalone/run_spm25.sh
echo "SPM hazır."

SPM hazır.


In [3]:
%%bash
set -e
chmod +x /content/spm/runtime_installer/Runtime_R2024b_for_spm_standalone_25.01.02.install
/content/spm/runtime_installer/Runtime_R2024b_for_spm_standalone_25.01.02.install \
  -agreeToLicense yes -mode silent
echo "MCR hazır."

MCR hazır.


---
## ⚙️ AYAR — Sadece BATCH_START'ı değiştir (Tab'a göre)

In [4]:
import os, math, glob, shutil
from pathlib import Path

# ============================================================
#  ↓↓↓  SADECE BU SATIRI DEĞİŞTİR  ↓↓↓
BATCH_START = 2200   # Tab 1→400 | Tab 2→1000 | Tab 3→1600
#  ↑↑↑  SADECE BU SATIRI DEĞİŞTİR  ↑↑↑
# ============================================================

INPUT_DIR  = Path("/content/drive/MyDrive/FINAL_DATABASE/ADNI_DATABASE/FINAL_DATASETS/ADNI_MRI_FINAL_NIFTI")
WORK       = Path("/content/work_adni")
DRIVE_OUT  = Path("/content/drive/MyDrive/FINAL_DATABASE/ADNI_DATABASE/PROCESSED/ADNI_MRI_SPM_PROCESSED")

N_JOBS     = 12    # 8'den 12'ye çıkarıldı
BATCH_SIZE = 600   # 400'den 600'e çıkarıldı

MNI_DIR = WORK / "MRI_mni"
JOB_DIR = WORK / "jobs"
LOG_DIR = WORK / "logs"
RC_DIR  = WORK / "rc"

for d in [MNI_DIR, JOB_DIR, LOG_DIR, RC_DIR, DRIVE_OUT]:
    d.mkdir(parents=True, exist_ok=True)

# Tüm input dosyaları
all_files = sorted(glob.glob(str(INPUT_DIR / "*.nii")))
print(f"Input toplam    : {len(all_files)} dosya")

# Bu batch aralığı
batch = all_files[BATCH_START : BATCH_START + BATCH_SIZE]
print(f"Batch aralığı   : [{BATCH_START} – {BATCH_START + len(batch) - 1}]  ({len(batch)} dosya)")

# Drive'da veya local'de biten dosyalar (çakışma koruması)
done_drive = {f.stem[3:] for f in DRIVE_OUT.glob("wr_*.nii")}
done_local = {f.stem[3:] for f in MNI_DIR.glob("wr_*.nii")}
done_set   = done_drive | done_local

pending = [f for f in batch if Path(f).stem not in done_set]

print(f"Zaten bitti     : {len(batch) - len(pending)} (atlandı)")
print(f"İşlenecek       : {len(pending)} dosya")

if not pending:
    print(f"\n✅ Bu batch bitti! Sonraki BATCH_START = {BATCH_START + BATCH_SIZE}")

Input toplam    : 3978 dosya
Batch aralığı   : [2200 – 2799]  (600 dosya)
Zaten bitti     : 0 (atlandı)
İşlenecek       : 600 dosya


In [5]:
import math

if not pending:
    print("Pending yok, job yazılmadı.")
else:
    # Eski job dosyalarını temizle
    for old in JOB_DIR.glob("job_*.m"):
        old.unlink()

    chunk = math.ceil(len(pending) / N_JOBS)

    job_template = r"""
% SPM25 Standalone — MRI Preprocessing Job {jid}
clc; clear; close all;

out_dir  = '{out_dir}';
ref_tmpl = fullfile(spm('Dir'),'canonical','avg152T1.nii');
tpm_path = fullfile(spm('Dir'),'tpm','TPM.nii');

spm('Defaults','fMRI'); spm_jobman('initcfg');
spm_get_defaults('cmdline', true);
spm_get_defaults('overwrite', true);

if ~exist(out_dir,'dir'), mkdir(out_dir); end

files = {{
{file_list}
}};

fprintf('JOB {jid}: %d dosya\n', numel(files));

for i = 1:numel(files)
    in_file = files{{i}};
    [~, base, ~] = fileparts(in_file);

    out_source = fullfile(out_dir, [base '.nii']);
    if ~exist(out_source,'file')
        copyfile(in_file, out_source);
    end
    [filepath, name, ~] = fileparts(out_source);

    wr_out = fullfile(filepath, ['wr_' name '.nii']);
    if exist(wr_out,'file')
        fprintf('SKIP: %s\n', base);
        continue;
    end

    clear matlabbatch;

    %% 1) COREGISTER ESTIMATE
    matlabbatch{{1}}.spm.spatial.coreg.estimate.ref    = {{ref_tmpl}};
    matlabbatch{{1}}.spm.spatial.coreg.estimate.source = {{out_source}};
    matlabbatch{{1}}.spm.spatial.coreg.estimate.other  = {{''}};
    matlabbatch{{1}}.spm.spatial.coreg.estimate.eoptions.cost_fun = 'nmi';
    matlabbatch{{1}}.spm.spatial.coreg.estimate.eoptions.sep  = [4 2];
    matlabbatch{{1}}.spm.spatial.coreg.estimate.eoptions.tol  = ...
        [0.02 0.02 0.02 0.001 0.001 0.001 0.01 0.01 0.01 0.001 0.001 0.001];
    matlabbatch{{1}}.spm.spatial.coreg.estimate.eoptions.fwhm = [7 7];

    %% 2) COREGISTER WRITE
    matlabbatch{{2}}.spm.spatial.coreg.write.ref = {{ref_tmpl}};
    matlabbatch{{2}}.spm.spatial.coreg.write.source = {{out_source}};
    matlabbatch{{2}}.spm.spatial.coreg.write.roptions.interp  = 4;
    matlabbatch{{2}}.spm.spatial.coreg.write.roptions.wrap    = [0 0 0];
    matlabbatch{{2}}.spm.spatial.coreg.write.roptions.mask    = 0;
    matlabbatch{{2}}.spm.spatial.coreg.write.roptions.prefix  = 'r_';

    r_vol = fullfile(filepath, ['r_' name '.nii,1']);

    %% 3) NEW SEGMENT  [OPTİMİZASYON: write=[0 0], samp=4]
    matlabbatch{{3}}.spm.spatial.preproc.channel.vols    = {{r_vol}};
    matlabbatch{{3}}.spm.spatial.preproc.channel.biasreg  = 0.001;
    matlabbatch{{3}}.spm.spatial.preproc.channel.biasfwhm = 60;
    matlabbatch{{3}}.spm.spatial.preproc.channel.write    = [0 0];

    for t = 1:6
        matlabbatch{{3}}.spm.spatial.preproc.tissue(t).tpm   = {{[tpm_path ',' num2str(t)]}};
        matlabbatch{{3}}.spm.spatial.preproc.tissue(t).ngaus  = 1;
        if t <= 3
            matlabbatch{{3}}.spm.spatial.preproc.tissue(t).native = [1 0];
        else
            matlabbatch{{3}}.spm.spatial.preproc.tissue(t).native = [0 0];
        end
        matlabbatch{{3}}.spm.spatial.preproc.tissue(t).warped = [0 0];
    end

    matlabbatch{{3}}.spm.spatial.preproc.warp.mrf    = 1;
    matlabbatch{{3}}.spm.spatial.preproc.warp.cleanup = 0;
    matlabbatch{{3}}.spm.spatial.preproc.warp.reg    = [0 0.001 0.1 0.05 0.2];
    matlabbatch{{3}}.spm.spatial.preproc.warp.affreg = 'mni';
    matlabbatch{{3}}.spm.spatial.preproc.warp.fwhm   = 0;
    matlabbatch{{3}}.spm.spatial.preproc.warp.samp   = 4;
    matlabbatch{{3}}.spm.spatial.preproc.warp.write  = [1 1];

    %% 4) NORMALISE WRITE
    def_field = fullfile(filepath, ['y_r_' name '.nii']);
    matlabbatch{{4}}.spm.spatial.normalise.write.subj.def      = {{def_field}};
    matlabbatch{{4}}.spm.spatial.normalise.write.subj.resample = {{fullfile(filepath, ['r_' name '.nii,1'])}};
    matlabbatch{{4}}.spm.spatial.normalise.write.woptions.bb   = [-90 -126 -72; 90 90 108];
    matlabbatch{{4}}.spm.spatial.normalise.write.woptions.vox  = [2 2 2];
    matlabbatch{{4}}.spm.spatial.normalise.write.woptions.interp = 4;

    try
        spm_jobman('run', matlabbatch);
        fprintf('[OK] %d/%d: %s\n', i, numel(files), base);
    catch ME
        fprintf('[FAIL] %s :: %s\n', base, ME.message);
    end

    if mod(i,20)==0
        fprintf('JOB {jid}: %d/%d\n', i, numel(files));
    end
end

fprintf('JOB {jid}: BITTI.\n');
exit;
"""

    jobs_written = 0
    for j in range(N_JOBS):
        part = pending[j*chunk : (j+1)*chunk]
        if not part:
            continue
        file_list = "\n".join([f"    '{p}';" for p in part])
        txt = job_template.format(
            jid=f"{j+1:02d}",
            out_dir=str(MNI_DIR),
            file_list=file_list
        )
        (JOB_DIR / f"job_{j+1:02d}.m").write_text(txt)
        jobs_written += 1

    print(f"✅ {jobs_written} job yazıldı — her biri ~{chunk} dosya")
    print(f"   Tahmini süre: ~{chunk * 2.0 / 60:.1f} saat (samp=4 ile daha hızlı)")

✅ 12 job yazıldı — her biri ~50 dosya
   Tahmini süre: ~1.7 saat (samp=4 ile daha hızlı)


In [6]:
%%bash
set -uo pipefail

SPM_RUN="/content/spm/spm_standalone/run_spm25.sh"
MCR="/usr/local/MATLAB/MATLAB_Runtime/R2024b"
WORK="/content/work_adni"
JOB_DIR="$WORK/jobs"
LOG_DIR="$WORK/logs"
RC_DIR="$WORK/rc"

mkdir -p "$LOG_DIR" "$RC_DIR"
rm -f "$WORK/pidmap.txt"

for m in "$JOB_DIR"/job_*.m; do
  b=$(basename "$m" .m)
  echo "[START] $b"
  "$SPM_RUN" "$MCR" script "$m" > "$LOG_DIR/$b.out.log" 2> "$LOG_DIR/$b.err.log" &
  echo "$! $b" >> "$WORK/pidmap.txt"
done

while read -r pid name; do
  wait "$pid"
  echo $? > "$RC_DIR/$name.rc"
  echo "[END] $name  rc=$(cat \"$RC_DIR/$name.rc\")"
done < "$WORK/pidmap.txt"

echo ""
echo "===== ÖZET ====="
for rc in "$RC_DIR"/job_*.rc; do
  b=$(basename "$rc" .rc)
  code=$(cat "$rc")
  [ "$code" = "0" ] && echo "[OK]   $b" || echo "[FAIL] $b → $LOG_DIR/$b.err.log"
done
echo ""
echo "wr_  tamamlanan: $(ls -1 /content/work_adni/MRI_mni/wr_*.nii  2>/dev/null | wc -l)"
echo "y_r_ tamamlanan: $(ls -1 /content/work_adni/MRI_mni/y_r_*.nii 2>/dev/null | wc -l)"

[START] job_01
[START] job_02
[START] job_03
[START] job_04
[START] job_05
[START] job_06
[START] job_07
[START] job_08
[START] job_09
[START] job_10
[START] job_11
[START] job_12
[END] job_01  rc=
[END] job_02  rc=
[END] job_03  rc=
[END] job_04  rc=
[END] job_05  rc=
[END] job_06  rc=
[END] job_07  rc=
[END] job_08  rc=
[END] job_09  rc=
[END] job_10  rc=
[END] job_11  rc=
[END] job_12  rc=

===== ÖZET =====
[OK]   job_01
[OK]   job_02
[OK]   job_03
[OK]   job_04
[OK]   job_05
[OK]   job_06
[OK]   job_07
[OK]   job_08
[OK]   job_09
[OK]   job_10
[OK]   job_11
[OK]   job_12

wr_  tamamlanan: 600
y_r_ tamamlanan: 600


cat: '"/content/work_adni/rc/job_01.rc"': No such file or directory
cat: '"/content/work_adni/rc/job_02.rc"': No such file or directory
cat: '"/content/work_adni/rc/job_03.rc"': No such file or directory
cat: '"/content/work_adni/rc/job_04.rc"': No such file or directory
cat: '"/content/work_adni/rc/job_05.rc"': No such file or directory
cat: '"/content/work_adni/rc/job_06.rc"': No such file or directory
cat: '"/content/work_adni/rc/job_07.rc"': No such file or directory
cat: '"/content/work_adni/rc/job_08.rc"': No such file or directory
cat: '"/content/work_adni/rc/job_09.rc"': No such file or directory
cat: '"/content/work_adni/rc/job_10.rc"': No such file or directory
cat: '"/content/work_adni/rc/job_11.rc"': No such file or directory
cat: '"/content/work_adni/rc/job_12.rc"': No such file or directory


In [ ]:
%%bash
# Ara dosyaları temizle (wr_ ve y_r_ dışındakiler)
OUT="/content/work_adni/MRI_mni"
find "$OUT" -maxdepth 1 -type f \( \
  -name "r_*.nii" -o -name "c*r_*.nii" -o -name "iy_r_*.nii" -o \
  -name "mr_*.nii" -o -name "m*.nii" -o \
  -name "*seg8*.mat" -o -name "*.ps" \) -delete
find "$OUT" -maxdepth 1 -type f -name "I*_T1.nii" -delete
echo "Temizlik tamam."
echo "wr_ : $(ls -1 $OUT/wr_*.nii  2>/dev/null | wc -l)"
echo "y_r_: $(ls -1 $OUT/y_r_*.nii 2>/dev/null | wc -l)"

Temizlik tamam.
wr_ : 600
y_r_: 600


In [7]:
import shutil
from pathlib import Path

MNI_DIR   = Path("/content/work_adni/MRI_mni")
DRIVE_OUT = Path("/content/drive/MyDrive/FINAL_DATABASE/ADNI_DATABASE/PROCESSED/ADNI_MRI_SPM_PROCESSED")
DRIVE_OUT.mkdir(parents=True, exist_ok=True)

copied_wr = copied_y = skipped = 0
for f in sorted(MNI_DIR.iterdir()):
    if not f.is_file() or f.suffix != '.nii':
        continue
    dst = DRIVE_OUT / f.name
    if dst.exists():
        skipped += 1
        continue
    shutil.copy2(f, dst)
    if f.name.startswith('wr_'):  copied_wr += 1
    elif f.name.startswith('y_r_'): copied_y += 1

print(f"Kopyalanan wr_  : {copied_wr}")
print(f"Kopyalanan y_r_ : {copied_y}")
print(f"Atlandı         : {skipped}")
print(f"\nDrive wr_  toplam : {len(list(DRIVE_OUT.glob('wr_*.nii')))}")
print(f"Drive y_r_ toplam : {len(list(DRIVE_OUT.glob('y_r_*.nii')))}")
print(f"\n➡ Sonraki BATCH_START = {BATCH_START + BATCH_SIZE}")

Kopyalanan wr_  : 600
Kopyalanan y_r_ : 600
Atlandı         : 0

Drive wr_  toplam : 2800
Drive y_r_ toplam : 2800

➡ Sonraki BATCH_START = 2800


In [8]:
# QC — Genel ilerleme özeti
from pathlib import Path

INPUT_DIR = Path("/content/drive/MyDrive/FINAL_DATABASE/ADNI_DATABASE/FINAL_DATASETS/ADNI_MRI_FINAL_NIFTI")
DRIVE_OUT = Path("/content/drive/MyDrive/FINAL_DATABASE/ADNI_DATABASE/PROCESSED/ADNI_MRI_SPM_PROCESSED")

total   = len(list(INPUT_DIR.glob("*.nii")))
done_wr = len(list(DRIVE_OUT.glob("wr_*.nii")))
done_y  = len(list(DRIVE_OUT.glob("y_r_*.nii")))

print(f"Toplam input  : {total}")
print(f"Biten (wr_)   : {done_wr}  ({done_wr/total*100:.1f}%)")
print(f"Biten (y_r_)  : {done_y}")
print(f"Kalan         : {total - done_wr}")
print(f"Sonraki start : {done_wr}")

# Eşleşme kontrolü
wr_set = {f.stem[3:] for f in DRIVE_OUT.glob('wr_*.nii')}
y_set  = {f.stem.replace('y_r_','',1) for f in DRIVE_OUT.glob('y_r_*.nii')}
orphan = wr_set - y_set
print(f"\n{'✅ Tüm wr_ için y_r_ eşi var.' if not orphan else f'⚠️  Eşsiz wr_: {len(orphan)} adet'}")

Toplam input  : 3978
Biten (wr_)   : 2800  (70.4%)
Biten (y_r_)  : 2800
Kalan         : 1178
Sonraki start : 2800

✅ Tüm wr_ için y_r_ eşi var.
